## 导入井头、井分层、井曲线数据


### 1) 环境与路径设置

确保可从仓库根目录导入 `src` 下的模块，并定位原始数据路径。


In [5]:
import sys
from pathlib import Path

import lasio
import pandas as pd

# 以 notebooks/ 为起点回到仓库根目录
repo_root = Path.cwd().resolve().parent
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from cup.petrel.load import (
    extract_gr_log_from_las,
    extract_rho_log_from_las,
    extract_vp_log_from_las,
    extract_vs_log_from_las,
    import_well_heads_petrel,
    import_well_tops_petrel,
)
from wtie.processing import grid

data_root = repo_root / "data"
well_heads_file = data_root / "raw" / "well_heads"
well_tops_file = data_root / "raw" / "well_tops"
las_dir = data_root / "vertical_well_las"

print(f"repo_root: {repo_root}")
print(f"well_heads_file exists: {well_heads_file.exists()}")
print(f"well_tops_file exists: {well_tops_file.exists()}")
print(f"las_dir exists: {las_dir.exists()}")

repo_root: C:\Users\WangQinZhuo\Program\libra_workflow_standardize
well_heads_file exists: True
well_tops_file exists: True
las_dir exists: True


### 2) 导入井头与井分层文件

使用 `load.py` 中现有函数读取并检查字段。


In [6]:
well_heads_df = import_well_heads_petrel(well_heads_file)
well_tops_df = import_well_tops_petrel(well_tops_file)

print("well_heads shape:", well_heads_df.shape)
print("well_tops shape:", well_tops_df.shape)

display(well_heads_df.head())
display(well_tops_df.head())

well_heads shape: (103, 7)
well_tops shape: (1045, 7)


,Name,Surface X,Surface Y,Well datum name,Well datum value,Bottom hole X,Bottom hole Y
0,TT6-2-1s,682381.23,3203910.90,KB,34.0,682808.39,3204854.15
1,TT6-2-1,682381.23,3203910.90,KB,34.0,682381.23,3203910.90
2,TT12-1-2,687479.84,3188728.43,KB,38.0,687479.84,3188728.43
3,TT12-1-1,689279.61,3184786.13,KB,38.0,689279.61,3184786.13
4,SHX36-5-A1,686186.78,3212086.34,KB,42.5,686675.77,3212095.53


,Well,Surface,X,Y,Z,MD,PVD auto
0,A1,H4-1,686332.8,3217090.3,-2431.9,2505.70,-2431.9
1,A1,H4-2,686332.3,3217085.0,-2448.4,2523.03,-2448.4
2,A1,H4-3,686331.7,3217078.2,-2469.8,2545.50,-2469.8
3,A1,H4-4,686330.9,3217070.0,-2495.4,2572.41,-2495.4
4,A1,H5-1,686329.5,3217057.2,-2535.6,2614.60,-2535.6


### 3) 批量导入 LAS 并构建含 Vp, Vs, Rho, GR 的 LogSet

遍历 `data/vertical_well_las` 下全部 `.las` 文件，逐井提取曲线并组装。


In [7]:
las_files = sorted(las_dir.glob("*.las"))
print(f"LAS files found: {len(las_files)}")
for p in las_files:
    print(" -", p.name)

logsets = {}
failed = []

for las_path in las_files:
    try:
        las_file = lasio.read(las_path)

        vp_log = extract_vp_log_from_las(las_file)
        vs_log = extract_vs_log_from_las(las_file)
        rho_log = extract_rho_log_from_las(las_file, curve_mnemonic="RHOZ")
        gr_log = extract_gr_log_from_las(las_file)

        logset = grid.LogSet({"Vp": vp_log, "Rho": rho_log})
        logset.add_log("Vs", vs_log)
        logset.add_log("GR", gr_log)

        # 以文件名（不含后缀）作为井名键
        logsets[las_path.stem] = logset

    except Exception as e:
        failed.append((las_path.name, str(e)))

print(f"\n成功构建 LogSet 数量: {len(logsets)}")
print(f"失败数量: {len(failed)}")
if failed:
    for name, msg in failed:
        print(f"[FAILED] {name}: {msg}")

LAS files found: 11
 - 2-ANP-2A-RJS-logs.las
 - L1-NW1-logs.las
 - L2-C1A-logs.las
 - L3-NW2A-logs.las
 - L4-C2-logs.las
 - L5-NW5-logs.las
 - L6-NW3A-logs.las
 - L9-NW4A-logs.las
 - NW13-logs.las
 - NW7-logs.las
 - NW8-logs.las

成功构建 LogSet 数量: 5
失败数量: 6
[FAILED] L2-C1A-logs.las: 未找到 Vp 曲线。候选简称: ['DT', 'DTC', 'DTCO', 'DTC1', 'AC']. 请检查是否存在其他可用简称？
[FAILED] L3-NW2A-logs.las: 未找到 Vp 曲线。候选简称: ['DT', 'DTC', 'DTCO', 'DTC1', 'AC']. 请检查是否存在其他可用简称？
[FAILED] L4-C2-logs.las: 未找到 Vp 曲线。候选简称: ['DT', 'DTC', 'DTCO', 'DTC1', 'AC']. 请检查是否存在其他可用简称？
[FAILED] L9-NW4A-logs.las: 未找到 Vp 曲线。候选简称: ['DT', 'DTC', 'DTCO', 'DTC1', 'AC']. 请检查是否存在其他可用简称？
[FAILED] NW13-logs.las: 未找到 Vp 曲线。候选简称: ['DT', 'DTC', 'DTCO', 'DTC1', 'AC']. 请检查是否存在其他可用简称？
[FAILED] NW8-logs.las: 指定的 Rho 曲线简称不存在: RHOZ. 可用曲线: ['DTC', 'DTS', 'GR', 'NPHI', 'PE', 'PERM', 'RHOB', 'VSH', 'RT10', 'RT20', 'RT30', 'RT60', 'RT90', 'POR', 'SWE']


### 4) 结果检查

检查每口井是否都包含 Vp、Vs、Rho、GR，并给出曲线长度与采样间隔。


In [8]:
summary_rows = []
for well_name, ls in logsets.items():
    summary_rows.append(
        {
            "well": well_name,
            "logs": list(ls.Logs.keys()),
            "n_samples": ls.Vp.size,
            "sampling_rate": ls.sampling_rate,
            "basis_type": ls.basis_type,
            "has_Vp": "Vp" in ls.Logs,
            "has_Vs": "Vs" in ls.Logs,
            "has_Rho": "Rho" in ls.Logs,
            "has_GR": "GR" in ls.Logs,
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("well").reset_index(drop=True)
display(summary_df)

if len(summary_df) > 0:
    all_ok = bool((summary_df[["has_Vp", "has_Vs", "has_Rho", "has_GR"]].all(axis=1)).all())
    print("全部井均含 Vp/Vs/Rho/GR:", all_ok)

# 示例查看第一口井的前几行
if logsets:
    first_well = sorted(logsets.keys())[0]
    print("\n示例井:", first_well)
    display(logsets[first_well].df.head())

,well,logs,n_samples,sampling_rate,basis_type,has_Vp,has_Vs,has_Rho,has_GR
0,2-ANP-2A-RJS-logs,"[Vp, Rho, Vs, GR]",32400,0.1250,MD (kb) [m],True,True,True,True
1,L1-NW1-logs,"[Vp, Rho, Vs, GR]",3818,0.1524,MD (kb) [m],True,True,True,True
2,L5-NW5-logs,"[Vp, Rho, Vs, GR]",4399,0.1524,MD (kb) [m],True,True,True,True
3,L6-NW3A-logs,"[Vp, Rho, Vs, GR]",4020,0.1524,MD (kb) [m],True,True,True,True
4,NW7-logs,"[Vp, Rho, Vs, GR]",2751,0.1524,MD (kb) [m],True,True,True,True


全部井均含 Vp/Vs/Rho/GR: True

示例井: 2-ANP-2A-RJS-logs


,Vp,Rho,Vs,GR
MD (kb) [m],,,,
2000.000,2294.935439,-971.142822,2428.576893,2.812
2000.125,2294.935439,-971.142822,2428.576893,2.812
2000.250,2294.935439,-971.142822,2428.576893,2.812
2000.375,2294.935439,-971.142822,2428.576893,2.812
2000.500,2294.935439,-971.142822,2428.576893,2.812
